In [1]:
import pandas as pd
import numpy as np
import os

# Cargar datos
ruta = os.path.join('..', 'Data', 'TouristAccommodationRaw26012026.csv')

df = pd.read_csv(ruta)

## 1. Limpieza y Preparación: Marketing y Estrategia Comercial

En esta fase, preparamos los datos para el análisis de mercado. El objetivo principal es normalizar las categorías y asegurar que la métrica de **precio** sea íntegra sin reducir el tamaño de la muestra (8000 registros).

* **Normalización de Texto:** Se eliminan espacios en blanco en las columnas `city` para evitar duplicidad de categorías.
* **Imputación de Precios:** Siguiendo la recomendación de no eliminar registros críticos, se aplica una **imputación por segmentos**.
    * **Lógica:** Se utiliza la **mediana** calculada por la combinación de ciudad y tipo de alojamiento. Esto garantiza que el precio asignado sea realista según su contexto geográfico y tipo de propiedad.
    * **Respaldo:** Se aplica una mediana global para cubrir cualquier caso excepcional sin datos segmentados.
* **Consistencia de Tipos:** Se fuerza el tipado a `float` para permitir cálculos estadísticos precisos.
* **Cuantificación de capacidad:** Con información de capacidad de huespedes y habitaciones, se crean una variable numérica (density) y otra categórica (density_category) como medidas posibles para capacidad de alojamientos.
* **Cuantificación de Equipamiento:** Se transforma la lista de texto de amenidades en una variable numérica (amenities_important_count) para facilitar el análisis de correlación con el precio.

In [2]:
# --- 1. NORMALIZACIÓN DE CATEGORÍAS ---

# Eliminamos espacios en blanco, estandarizamos a minúsculas y capitalizamos
# (ej. "madrid " -> "Madrid")
df['city'] = df['city'].str.strip().str.capitalize()

# Limpiamos los tipos de alojamiento para asegurar agrupaciones precisas en la imputación
df['room_type'] = df['room_type'].str.strip()

In [3]:
# --- 2. IMPUTACIÓN DE PRECIOS ---

# Paso A: Imputación segmentada (Por Ciudad y Tipo de Habitación)
# La mediana por segmento es más robusta que la media ante valores atípicos
df['price'] = df.groupby(['city', 'room_type'])['price'].transform(
    lambda x: x.fillna(x.median())
)

# Paso B: Imputación de seguridad
# En caso de que un segmento completo sea nulo, usamos la mediana de todo el dataset
df['price'] = df['price'].fillna(df['price'].median())

# Paso C: Aseguramos que price sea numérico para el análisis comercial
df['price'] = df['price'].astype(float)

In [4]:
# --- 3. LIMPIEZA DE PUNTUACIÓN DE ZONA ---

# Conversión de la puntuación de ubicación
df['review_scores_location'] = pd.to_numeric(df['review_scores_location'], errors='coerce')

In [5]:
# --- 4. LIMPIEZA Y TRANSFORMACION DE CAPACIDAD ---

# PASO A: Conversión masiva a formato numérico de las variables de capacidad
columnas_capacidad = ['accommodates', 'bedrooms', 'beds']

for col in columnas_capacidad:
    df[col] = pd.to_numeric(df[col], errors='coerce')


# PASO B: Creamos una columna de densidad: capacidad de huespedes dividida por capacidad de habitaciones
df['density'] = df['accommodates'] / df['bedrooms'].replace(0, 1)


# PASO C: Creamos otra columna que transforma la densidad a categorica
# (1) Definimos las condiciones

conditions = [
    (df['density'] < 2),
    (df['density'] == 2), # Más de la midad de los casos
    (df['density'] > 2)
]

# (2) Definimos las etiquetas correspondientes
choices = ['<2', '2', '>2']

# (3) Creamos la nueva columna
df['density_category'] = np.select(conditions, choices, default=None)

In [6]:
# --- 5. PROCESAMIENTO DE AMENIDADES ---

# Cuantificamos la oferta de servicios importantes: transformamos la lista de texto en un conteo numérico
# Si el campo es nulo, se asigna nulo igual

# PASO A: Identificamos 5 grupos importantes de amenidades, segun informes sobre alojamientos turisticos (Booking.com, etc.)
# Las amenidades son identificados con todas las amenidades distintas en la columna "amenities_list"
category_mapping = {
    'ac_heating': [
        'air conditioning', 'central air conditioning', 'heating', 'central heating', 
        'portable heater', 'portable fans', 'heated floors', 'indoor fireplace'],
    'internet': [
        'wifi', 'wifi u2013 100 mbps', 'internet', 'wireless internet', 'ethernet connection', 
        'pocket wifi'],
    'outdoor_space': [
        'patio or balcony', 'balcony', 
        'terrace', 'garden or backyard', 'shared garden or backyard', 'outdoor furniture', 
        'outdoor seating', 'outdoor dining area'],
    'parking': [
        'parking', 'free parking on premises', 'free parking on street', 'free street parking', 
        'free driveway parking on premises u2013 1 space', 'paid parking on premises', 
        'paid parking off premises', 'paid parking garage on premises', 
        'paid parking garage off premises', 'ev charger'],
    'view_scenery': [
        'beach view', 'beachfront', 'lake access', 'mountain view', 'waterfront'] 
}


# PASO B: funcion para contar cuantos tipos distintos de amenidades hay
def get_important_amenities_count(row_amenities):
    if pd.isna(row_amenities):
        return np.nan
    
    # Normalización: convertimos la lista del alojamiento a minúsculas y quitamos espacios
    current_amenities = [a.strip().lower() for a in str(row_amenities).split(',')]
    
    count = 0
    for category, items in category_mapping.items():
        # Verificamos si al menos uno de los ítems de la categoría está presente
        if any(item in current_amenities for item in items):
            count += 1
            
    return count


# PASO C: Aplicamos la lógica al DataFrame
df['amenities_important_count'] = df['amenities_list'].apply(get_important_amenities_count)

## 2. Limpieza para Experiencia del Cliente.

Para responder a las preguntas sobre satisfacción sin reducir la muestra total de 8000 registros, se aplica la siguiente lógica:

* **Tratamiento de Ratings:** Se transforman a formato numérico, manteniendo los valores ausentes como `NaN`.

In [7]:
# --- LIMPIEZA PARA EXPERIENCIA DEL CLIENTE ---

# Definición de las columnas que responden a la pregunta de negocio del analista
columnas_rating = [
    'review_scores_rating',        # Evaluación general
    'review_scores_accuracy',      # Precisión de detalles
    'review_scores_cleanliness',   # Higiene
    'review_scores_checkin',       # Proceso de entrada
    'review_scores_communication'  # Comunicación
]

# Convertir a numérico. Los errores o celdas vacías se convierten en NaN.
for col in columnas_rating:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## 3. Normalización Temporal y Validación de Disponibilidad

* **Procesamiento de Fechas:** * Conversión de `insert_date` a formato `datetime` mediante coerción de errores.
    * Generación de la columna `month` (Periodo) para análisis de series temporales.
* **Estandarización de Disponibilidad:** * Mapeo de valores `"VERDADERO"` y `"NULL"` a lógica booleana.
    * Conversión final de la columna `has_availability` a tipo `bool` para optimizar el filtrado y el almacenamiento.

In [8]:
# --- 1. NORMALIZACIÓN TEMPORAL ---
# Conversión a datetime: 'errors="coerce"' transforma formatos inválidos en NaT
df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")

# Extracción del periodo mensual para facilitar agregaciones temporales
df["month"] = df["insert_date"].dt.to_period("M")

# --- 2. VALIDACIÓN DE DISPONIBILIDAD ---
# Homogeneización de valores categóricos a booleanos y manejo de nulos
df["has_availability"] = df["has_availability"].replace({
    "VERDADERO": True, 
    "NULL": False
}).fillna(False).astype(bool)

C:\Users\Jianji Chen\AppData\Local\Temp\ipykernel_71136\1242647963.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")
C:\Users\Jianji Chen\AppData\Local\Temp\ipykernel_71136\1242647963.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(False).astype(bool)


OPERACIONES Y GESTION DE INVENTARIO 

 TRANSFORMACIÓN: COLUMNA is_instant_bookable A TIPO BOOLEANO   
Convertir valores de texto español (FALSO/VERDADERO) a tipo bool para optimizar memoria y facilitar operaciones lógicas

In [9]:
# Mapeo de valores españoles a booleanos
df['is_instant_bookable'] = df['is_instant_bookable'].map({
    'VERDADERO': True,
    'FALSO': False
})

Extracción de componentes temporales desde la columna 'month' (dtype: period[M])
para análisis de series de tiempo y segmentación estacional.

Variables creadas:
- year (int): Año calendario en formato numérico
- month_new (str): Mes del año en formato ISO 3-letter code 

In [10]:
df["year"] = df['month'].dt.year    

# Para el mes, extraemos el número y lo mapeamos
mes_ingles = {
    1: 'JAN', 2: 'FEB', 3: 'MAR', 4: 'APR',
    5: 'MAY', 6: 'JUN', 7: 'JUL', 8: 'AUG',
    9: 'SEP', 10: 'OCT', 11: 'NOV', 12: 'DEC'
}

df['month_new'] = df['month'].dt.month.map(mes_ingles)

## 4. Crear una columna de indice para identificar el id unico de registros

In [13]:
# Este columna de indice 'unique_id' con valor empieza desde 1 hasta el numero de registros
df['unique_id'] = np.arange(1, len(df)+1)

df.columns

Index(['apartment_id', 'name', 'description', 'host_id', 'neighbourhood_name',
       'neighbourhood_district', 'room_type', 'accommodates', 'bathrooms',
       'bedrooms', 'beds', 'amenities_list', 'price', 'minimum_nights',
       'maximum_nights', 'has_availability', 'availability_30',
       'availability_60', 'availability_90', 'availability_365',
       'number_of_reviews', 'first_review_date', 'last_review_date',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'is_instant_bookable', 'reviews_per_month',
       'country', 'city', 'insert_date', 'density', 'density_category',
       'amenities_important_count', 'month', 'year', 'month_new', 'unique_id'],
      dtype='object')

## 5. Exportación de Resultados

Una vez finalizado el proceso de limpieza y normalización para los tres departamentos se procede a exportar el **Dataset Limpio**.

* **Ruta de destino:** Se almacena en la carpeta institucional `/Data/` bajo el nombre `TouristAccommodationClean19012026.csv`.
* **Codificación:** Se utiliza `utf-8-sig` para garantizar que la corrección de caracteres especiales.

In [14]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_archivo_limpio = 'TouristAccommodationClean26012026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_guardado = os.path.join('..', 'Data', nombre_archivo_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números
# encoding='utf-8-sig' 
df.to_csv(ruta_guardado, index=False, encoding='utf-8-sig')